# FibRec, FibMemo and FibIt (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_python.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `FibRec`, `FibMemo` and `FibIt`: the algorithms in Python, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/FibRec_FibMemo_FibIt_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fib_rec` (snake_case).

## Fibonacci: FibRec, FibMemo and FibIt (overlapping subproblems)

*Input*: a natural number $n$. *Output*: the Fibonacci number $F(n)$, with $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — direct translation of the definition: the subproblems overlap and the calls are $2F(n+1)-1=\Theta(\varphi^n)$, exponential; the depth is $n$.
- **FibMemo**$(n,memo)$ — recursion with memoization: every value is computed once (the vector `memo` is $-1$ for "undefined"); $2n-1$ calls, memory $\Theta(n)$.
- **FibIt**$(n)$ — loop with two variables: $n-1$ iterations, memory $\Theta(1)$.

On the example $\texttt{FibRec}(5)$ of the chapter we count how many times each subproblem is called (the tree has $15$ nodes, but only $6$ different values are needed). Then the table of the counts (which coincide with the values of the chapter: $21\,891$, $242\,785$, $2\,692\,537$ calls for $n=20,25,30$) and the measured times.

In [1]:
import time

fib_calls = 0   # calls executed

def fib_rec(n):
    """FibRec(n)"""
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    return fib_rec(n - 1) + fib_rec(n - 2)

def fib_memo(n, memo):
    """FibMemo(n, memo): memo[k] = -1 means "undefined"."""
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    if memo[n] == -1:
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

def fib_it(n):
    """FibIt(n)"""
    if n <= 1:
        return n
    prev = 0                        # F(i-2)
    curr = 1                        # F(i-1)
    i = 2
    while i <= n:
        nxt = prev + curr
        prev = curr
        curr = nxt
        i = i + 1
    return curr

def fib_rec_hist(n, counts):
    """Like FibRec, but counts the calls for each value of the parameter."""
    counts[n] += 1
    if n <= 1:
        return n
    return fib_rec_hist(n - 1, counts) + fib_rec_hist(n - 2, counts)

counts = [0] * 6
print("FibRec(5) =", fib_rec_hist(5, counts))
print("calls for each value of the parameter:", "  ".join(f"{k}:{counts[k]}" for k in range(5, -1, -1)), f"  total {sum(counts)}")
print()
print(f"{'n':>3s} {'F(n)':>8s} {'FibRec':>9s} {'2F(n+1)-1':>10s} {'FibMemo':>8s} {'FibIt':>6s}  (calls, calls, calls, iterations)")
for n in [10, 15, 20, 25, 30]:
    f = fib_it(n)
    fib_calls = 0
    r1 = fib_rec(n)
    c1 = fib_calls
    fib_calls = 0
    r2 = fib_memo(n, [-1] * (n + 1))
    c2 = fib_calls
    ok = "OK" if (r1 == r2 == f and c1 == 2 * fib_it(n + 1) - 1 and c2 == 2 * n - 1) else "ERROR"
    print(f"{n:>3d} {f:>8d} {c1:>9d} {2 * fib_it(n + 1) - 1:>10d} {c2:>8d} {n - 1:>6d}  {ok}")
print()
print("Measured times (they depend on the machine):")
for n in [20, 25, 30]:
    t0 = time.perf_counter(); fib_rec(n); t1 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter()
    for _ in range(1000):
        fib_it(n)
    t2 = (time.perf_counter() - t0) * 1000 / 1000
    print(f"n={n:<3d} FibRec {t1:>10.4f} ms   FibIt {t2:>10.6f} ms")

FibRec(5) = 5
calls for each value of the parameter: 5:1  4:1  3:2  2:3  1:5  0:3   total 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (calls, calls, calls, iterations)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Measured times (they depend on the machine):
n=20  FibRec     2.0071 ms   FibIt   0.001060 ms
n=25  FibRec    22.4404 ms   FibIt   0.001361 ms
n=30  FibRec   244.0533 ms   FibIt   0.001460 ms


**Try it yourself.** Increase $n$ (with $n=32$ `FibRec` makes $7\,049\,155$ calls, `FibMemo` $63$) and see how the time of `FibRec` grows.

In [2]:
# Increase n: with n = 32 FibRec makes 7 049 155 calls, FibMemo 63.
n = 32
fib_calls = 0
print(fib_rec(n), fib_calls)
fib_calls = 0
print(fib_memo(n, [-1] * (n + 1)), fib_calls)

2178309 7049155
2178309 63


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).